# Activation Distributions per Layer

Inspect whether activations are **saturated**, **dead** (ReLU), or well-scaled.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Model with hooks

In [ ]:
class MLPAct(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(16, 64)
        self.l2 = nn.Linear(64, 32)
        self.l3 = nn.Linear(32, 3)
    def forward(self, x):
        self.a1 = F.relu(self.l1(x))
        self.a2 = F.relu(self.l2(self.a1))
        return self.l3(self.a2)

model = MLPAct()
X = torch.randn(500, 16)
with torch.no_grad():
    model(X)

## 2. Histogram per layer

In [ ]:
acts = [('layer1', model.a1), ('layer2', model.a2)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, a) in zip(axes, acts):
    ax.hist(a.flatten().numpy(), bins=50, color='steelblue', edgecolor='black', alpha=0.85)
    ax.set_title(f'{name} activations'); ax.set_xlabel('value')
plt.tight_layout(); plt.show()

## 3. Compare before/after BatchNorm-style scaling

In [ ]:
raw = model.a1.flatten()
scaled = (raw - raw.mean()) / (raw.std() + 1e-5)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(raw.numpy(), bins=50, color='coral'); axes[0].set_title('Raw ReLU activations')
axes[1].hist(scaled.numpy(), bins=50, color='seagreen'); axes[1].set_title('Standardized (illustrative)')
plt.tight_layout(); plt.show()

## 4. Dead ReLU fraction per neuron

In [ ]:
dead_frac = (model.a1 == 0).float().mean(dim=0).numpy()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(dead_frac)), dead_frac, color='gray', edgecolor='black')
ax.set_xlabel('neuron index'); ax.set_ylabel('fraction zero')
ax.set_title('Dead ReLU rate per neuron (one batch)')
plt.tight_layout(); plt.show()

## 5. Activation mean/std per layer

In [ ]:
stats = {'L1': model.a1, 'L2': model.a2}
means = [t.mean().item() for t in stats.values()]
stds = [t.std().item() for t in stats.values()]
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(2); w = 0.35
ax.bar(x - w/2, means, w, label='mean', color='steelblue')
ax.bar(x + w/2, stds, w, label='std', color='orange')
ax.set_xticks(x); ax.set_xticklabels(list(stats.keys()))
ax.legend(); ax.set_title('Activation statistics')
plt.tight_layout(); plt.show()